# Handwritten Notes Maker
Turns a Markdown file (`.md`) into a hand-drawn notebook PDF: spiral binding, wobbly ink boxes, clouds, sticky notes, flow diagrams.

**How to use**
1. Put this notebook, your `.md` file and nothing else in one folder.
2. In cell 2 (Settings), set `INPUT_MD` and `OUTPUT_PDF`.
3. Run all cells (first run installs packages, Chromium and fonts; later runs take seconds).

**Cheat sheet for the `.md` file** (full demo: `notes_format_guide.md`)
| Write | Get |
|---|---|
| front matter `title:` `subtitle:` `tagline:` `chips:` `footer:` | cover page + contents + default footer |
| `# Title` | new page with the boxed title |
| `<!-- section: Name -->` | groups pages in the contents |
| `## Heading` | section heading (inks rotate) |
| `==text==` · `[[Ctrl]]` · `{r:text}` (r b t g p a) · `:bulb:` | highlighter · key cap · coloured ink · doodle icon |
| `> tip:` `note` `warning` `example` `info` `remember` `ai` `cost` `pro` `try` | callout box (add `"Title"` before the colon) |
| `> takeaway:` · `> sticky:` · `> quote:` · `> footer:` | cloud · sticky note · quote · page footer |
| `::: cols` with `---` between columns | columns (outer block needs `::::` if it contains another block) |
| `::: grid 3` · `::: box blue "Title" :star:` · `::: steps` · `::: qa` | grid · box · numbered step cards · 2-column Q&A |
| `::: flow caption` then `*[clock] A -> [mail] B \| [hash] C` | n8n-style flow diagram (`*` = trigger, `label:` on arrows) |
| `- [x]` / `- [ ]` | ticked / empty checkbox |
| `<!-- sub: text -->` under a title | small subtitle line under the numbered title |
| `((text))` `((r:text))` | result chip (green by default) |
| `> plain:` | purple "In plain English" line |
| ```` ```sql ```` `dax` `python` `js` | code with coloured keywords, functions, values, comments |
| `::: parts` (`text \| caption \| colour`) · `::: chain` / `::: chain v` | labelled statement pieces · stage boxes with arrows |
| `{k:text}` · `::: box pink` | pink ink and box (also `nofill`) |

**Automatic**: `AUTO_COLOUR` (cell 2) colours plain tables, bullets, headings, boxes and untagged code. Block nesting colons are fixed for you.


In [1]:
# ── 1. SETUP: installs anything missing (safe to re-run) ─────────────────────
import sys, subprocess, importlib, os

PACKAGES = {  # import name : pip name
    "markdown_it": "markdown-it-py",
    "mdit_py_plugins": "mdit-py-plugins",
    "bs4": "beautifulsoup4",
    "yaml": "pyyaml",
    "playwright": "playwright",
}

def _pip(pkg):
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(cmd + ["--break-system-packages"])

for mod, pkg in PACKAGES.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"Installing {pkg} ...")
        _pip(pkg)

# Playwright needs its own Chromium once (about 150 MB, downloaded on first run only)
if not os.environ.get("NOTES_SKIP_BROWSER_INSTALL"):
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=False)
print("Setup done.")


Installing mdit-py-plugins ...
Installing beautifulsoup4 ...
Installing playwright ...
Setup done.


In [ ]:
# ── 2. SETTINGS: the only cell you normally edit ─────────────────────────────
from pathlib import Path

BASE       = Path.cwd()                      # folder this notebook lives in
INPUT_MD   = BASE / "sql_notes.md"           # your content file
OUTPUT_PDF = BASE / "sql_notes.pdf"          # where the PDF goes
FONT_DIR   = BASE / "fonts"                  # fonts are downloaded here once

MY_NOTES_AREA = False     # True = fill empty space at the bottom of a page with ruled "my notes" lines
TITLE_STYLE   = "modern"  # "modern" = number badge + part label + colourful title | "banner" = old boxed title
AUTO_COLOUR   = True      # colours plain Markdown for you: table headers, bullets, headings, boxes, untagged code
RULED_PAPER   = True      # faint notebook lines and a red margin line behind every page
FILL_BOXES    = True      # soft colour fill inside boxes, callouts and code blocks
SYNTAX_COLOURS = True     # colour keywords, functions, text values and comments in ```sql / ```dax / ```python code
PAGE_LABEL    = "page {n} of {total}"   # bottom-right page label ("" = none)
PREVIEW_PAGES = []        # e.g. [1, 5] saves PNG previews of those pages to check the look
CHROMIUM_PATH = os.environ.get("CHROMIUM_PATH")   # leave as is (None = Playwright's own browser)

# Ink colours (light, vibrant, readable on white)
INK = {
    "ink":    "#262626",   # main pen
    "blue":   "#1d4ed8",
    "teal":   "#0f766e",
    "red":    "#c2410c",
    "green":  "#15803d",
    "purple": "#7c3aed",
    "amber":  "#b45309",
    "pink":   "#be185d",
}
STICKY_COLOURS = ["#fff59d", "#ffd6e7", "#d9f2ff", "#dcfce7"]   # sticky notes rotate through these


In [3]:
# ── 3. FONTS: Kalam handwriting + a clearer capital "I" (so I2 never reads as 12)
import base64, urllib.request

FONT_FILES = {
    "k300": "kalam/files/kalam-latin-300-normal.woff2",
    "k400": "kalam/files/kalam-latin-400-normal.woff2",
    "k700": "kalam/files/kalam-latin-700-normal.woff2",
    "d400": "delius/files/delius-latin-400-normal.woff2",
}
CDNS = ["https://cdn.jsdelivr.net/npm/@fontsource/", "https://unpkg.com/@fontsource/"]

def font_b64(key):
    FONT_DIR.mkdir(exist_ok=True)
    rel = FONT_FILES[key]
    path = FONT_DIR / rel.split("/")[-1]
    if not path.exists():
        print("Downloading font", path.name)
        for cdn in CDNS:
            try:
                urllib.request.urlretrieve(cdn + rel, path); break
            except Exception as e:
                print("  could not reach", cdn, "->", e)
        else:
            raise RuntimeError("Font download failed: put the .woff2 files into the fonts folder manually")
    return base64.b64encode(path.read_bytes()).decode()

FONTS_B64 = {k: font_b64(k) for k in FONT_FILES}
print("Fonts ready.")


Fonts ready.


In [4]:
# ── 4. DOODLE ICONS: use in Markdown as :name: (e.g. :bulb: :robot: :mail:) ──
ICONS = {
 "bulb":"M9 18h6M10 21h4M12 3a6 6 0 0 0-3.5 10.9c.6.5 1 1.2 1 2V17h5v-1.1c0-.8.4-1.5 1-2A6 6 0 0 0 12 3z",
 "warn":"M12 3l10 18H2zM12 10v5M12 18v.4",
 "star":"M12 2.5l2.9 6 6.6.8-4.9 4.5 1.3 6.5L12 17l-5.9 3.3 1.3-6.5L2.5 9.3l6.6-.8z",
 "check":"M12 3a9 9 0 1 0 .01 0M8 12.5l3 3 5-6",
 "cross":"M12 3a9 9 0 1 0 .01 0M9 9l6 6M15 9l-6 6",
 "keyboard":"M4.5 6h15A2.5 2.5 0 0 1 22 8.5v8a2.5 2.5 0 0 1-2.5 2.5h-15A2.5 2.5 0 0 1 2 16.5v-8A2.5 2.5 0 0 1 4.5 6zM6 10h1M10 10h1M14 10h1M18 10h1M6 14h1M18 14h1M9 14h6",
 "gear":"M12 9a3 3 0 1 0 .01 0M12 2v3M12 19v3M2 12h3M19 12h3M4.9 4.9l2.1 2.1M17 17l2.1 2.1M4.9 19.1L7 17M17 7l2.1-2.1",
 "bolt":"M13 2L4 14h7l-1 8 9-12h-7z",
 "robot":"M7 8h10a3 3 0 0 1 3 3v6a3 3 0 0 1-3 3H7a3 3 0 0 1-3-3v-6a3 3 0 0 1 3-3zM12 4v4M9 13h.01M15 13h.01M9.5 17h5M12 2.5a1 1 0 1 0 .01 0",
 "ai":"M12 3l1.8 4.7L18.5 9.5l-4.7 1.8L12 16l-1.8-4.7L5.5 9.5l4.7-1.8zM19 15l.8 2 2 .8-2 .8-.8 2-.8-2-2-.8 2-.8z",
 "cloud":"M7 18a4 4 0 0 1-.5-8A6 6 0 0 1 18 9a4.5 4.5 0 0 1-.5 9z",
 "server":"M5 4h14a2 2 0 0 1 2 2v3a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V6a2 2 0 0 1 2-2zM5 13h14a2 2 0 0 1 2 2v3a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-3a2 2 0 0 1 2-2zM7 7.5h.01M7 16.5h.01",
 "db":"M5 5.5c0 1.4 3.1 2.5 7 2.5s7-1.1 7-2.5S15.9 3 12 3 5 4.1 5 5.5zM5 5.5v13c0 1.4 3.1 2.5 7 2.5s7-1.1 7-2.5v-13M5 12c0 1.4 3.1 2.5 7 2.5s7-1.1 7-2.5",
 "sheet":"M6 3h12a2 2 0 0 1 2 2v14a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2zM4 9h16M4 15h16M10 3v18",
 "mail":"M5 5h14a2 2 0 0 1 2 2v10a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V7a2 2 0 0 1 2-2zM3 7l9 6 9-6",
 "chat":"M21 12a8 8 0 0 1-11.6 7.1L4 20l1.1-4.6A8 8 0 1 1 21 12z",
 "hash":"M5 9h15M4 15h15M10 3L8 21M16 3l-2 18",
 "webhook":"M6 14.5a2.5 2.5 0 1 0 .01 0M18 14.5a2.5 2.5 0 1 0 .01 0M12 3.5a2.5 2.5 0 1 0 .01 0M10.8 8.2L7.4 14.8M8.5 17h7M13.2 8.2l3.4 6.6",
 "clock":"M12 3a9 9 0 1 0 .01 0M12 7v5l3 2",
 "calendar":"M5 5h14a2 2 0 0 1 2 2v12a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V7a2 2 0 0 1 2-2zM3 10h18M8 3v4M16 3v4",
 "code":"M8 8l-4 4 4 4M16 8l4 4-4 4M14 5l-4 14",
 "link":"M10 14a4 4 0 0 0 5.7 0l3-3a4 4 0 0 0-5.7-5.7l-1 1M14 10a4 4 0 0 0-5.7 0l-3 3a4 4 0 0 0 5.7 5.7l1-1",
 "lock":"M7 11h10a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H7a2 2 0 0 1-2-2v-6a2 2 0 0 1 2-2zM8 11V7a4 4 0 0 1 8 0v4",
 "money":"M4 6h16a2 2 0 0 1 2 2v8a2 2 0 0 1-2 2H4a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2zM12 9a3 3 0 1 0 .01 0M6 9v.01M18 15v.01",
 "euro":"M17 6a7 7 0 1 0 0 12M4 10h9M4 14h9",
 "chart":"M3 20h18M6 20v-6M10.5 20V9M15 20v-9M19.5 20V5",
 "user":"M12 4a4 4 0 1 0 .01 0M4 21a8 8 0 0 1 16 0",
 "users":"M9 4.5a3.5 3.5 0 1 0 .01 0M2 20a7 7 0 0 1 14 0M17 6.5a2.5 2.5 0 1 0 .01 0M17 14a5 5 0 0 1 5 5",
 "rocket":"M14 4c3 0 6 3 6 6l-6 6-4-4zM15.5 7.1a1.4 1.4 0 1 0 .01 0M10 12l-4 1-2 3 4-1M12 14l-1 4-3 2 1-4M5 19l-2 2",
 "trophy":"M8 4h8v5a4 4 0 0 1-8 0zM8 6H5a3 3 0 0 0 3 4M16 6h3a3 3 0 0 1-3 4M12 13v4M9 20h6M10 17h4",
 "heart":"M12 20s-7-4.4-7-10a4 4 0 0 1 7-2.6A4 4 0 0 1 19 10c0 5.6-7 10-7 10z",
 "book":"M4 5a2 2 0 0 1 2-2h14v16H6a2 2 0 0 0-2 2zM4 19V5M9 8h7M9 11h5",
 "flag":"M5 21V4M5 4h11l-2 4 2 4H5",
 "target":"M12 3a9 9 0 1 0 .01 0M12 7a5 5 0 1 0 .01 0M12 11a1 1 0 1 0 .01 0",
 "globe":"M12 3a9 9 0 1 0 .01 0M3 12h18M12 3c3 3 3 15 0 18M12 3c-3 3-3 15 0 18",
 "form":"M6 3h12a2 2 0 0 1 2 2v14a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2zM8 8h8M8 12h8M8 16h5",
 "filter":"M3 4h18l-7 8v6l-4 2v-8z",
 "branch":"M6 3a2 2 0 1 0 .01 0M6 17a2 2 0 1 0 .01 0M18 7a2 2 0 1 0 .01 0M6 7v10M6 13c0-3 4-4 10-4",
 "loop":"M17 2l3 3-3 3M4 11V9a4 4 0 0 1 4-4h12M7 22l-3-3 3-3M20 13v2a4 4 0 0 1-4 4H4",
 "merge":"M6 3v6c0 4 6 5 6 9v3M18 3v6c0 4-6 5-6 9",
 "play":"M12 3a9 9 0 1 0 .01 0M10 8l6 4-6 4z",
 "wait":"M6 3h12M6 21h12M7 3c0 6 10 6 10 18M17 3c0 6-10 6-10 18",
 "bug":"M12 7a5 5 0 0 1 5 5v3a5 5 0 0 1-10 0v-3a5 5 0 0 1 5-5zM12 7v13M9 4l1.5 2M15 4l-1.5 2M3 11h4M17 11h4M3 17h4M17 17h4",
 "shield":"M12 3l8 3v6c0 5-3.5 8-8 9-4.5-1-8-4-8-9V6zM9 12l2 2 4-4",
 "file":"M6 3h8l5 5v13H6zM14 3v5h5",
 "search":"M10 3.5a6.5 6.5 0 1 0 .01 0M15 15l6 6",
 "pin":"M12 21s-7-6.5-7-12a7 7 0 0 1 14 0c0 5.5-7 12-7 12zM12 6.5a2.5 2.5 0 1 0 .01 0",
 "box":"M3 7l9-4 9 4v10l-9 4-9-4zM3 7l9 4 9-4M12 11v10",
 "plug":"M9 2v5M15 2v5M6 7h12v4a6 6 0 0 1-12 0zM12 17v5",
 "brain":"M9 4a3 3 0 0 0-3 3 3 3 0 0 0-2 5 3 3 0 0 0 2 5 3 3 0 0 0 6 1V5a3 3 0 0 0-3-1zM15 4a3 3 0 0 1 3 3 3 3 0 0 1 2 5 3 3 0 0 1-2 5 3 3 0 0 1-6 1",
 "pencil":"M4 20l4-1 11-11-3-3L5 16zM14 6l3 3",
 "list":"M9 6h11M9 12h11M9 18h11M4 6h.01M4 12h.01M4 18h.01",
 "phone":"M9 2h6a2 2 0 0 1 2 2v16a2 2 0 0 1-2 2H9a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2zM11 18h2",
 "cart":"M3 4h2l2.5 11h11l2-8H6.5M9 18.5a1.5 1.5 0 1 0 .01 0M17 18.5a1.5 1.5 0 1 0 .01 0",
 "coffee":"M4 9h12v6a5 5 0 0 1-5 5H9a5 5 0 0 1-5-5zM16 11h2a2 2 0 0 1 0 4h-2M8 3v3M12 3v3",
 "question":"M12 3a9 9 0 1 0 .01 0M9.5 9.5a2.5 2.5 0 1 1 3.5 2.3c-.6.3-1 .9-1 1.6V14M12 17.5v.4",
 "eye":"M2 12s4-7 10-7 10 7 10 7-4 7-10 7S2 12 2 12zM12 9a3 3 0 1 0 .01 0",
 "wrench":"M14.5 6.5a4 4 0 0 0-5 5L3 18l3 3 6.5-6.5a4 4 0 0 0 5-5l-2.5 2.5-2.5-.5-.5-2.5z",
 "refresh":"M20 11a8 8 0 0 0-14.5-4.5L4 8M4 4v4h4M4 13a8 8 0 0 0 14.5 4.5L20 16M20 20v-4h-4",
 "send":"M22 2L11 13M22 2l-7 20-4-9-9-4z",
 "learn":"M2 9l10-5 10 5-10 5zM6 11v5c3 2 9 2 12 0v-5M22 9v6",
 "sparkle":"M12 3v18M3 12h18M6 6l12 12M18 6L6 18",
 "swirl":"M12 12a2 2 0 1 1 2 2 4 4 0 1 1-4-4 6 6 0 1 1 6 6",
 "arrow":"M4 18C8 8 14 6 20 8M15 4l5 4-5 3",
 "key":"M7.5 11a4.5 4.5 0 1 0 .01 0M11.5 13.5L21 4M17 8l3 3M15 10l2 2",
 "table":"M4 4h16v16H4zM4 10h16M4 15h16M10 4v16",
 "split":"M4 12h6M10 12l8-6M10 12l8 6M18 4l2 2-2 2M18 16l2 2-2 2",
 "stop":"M8 3h8l5 5v8l-5 5H8l-5-5V8zM9 9l6 6M15 9l-6 6",
 "home":"M3 11l9-8 9 8M5 9.5V21h14V9.5",
 "cursor":"M5 3l14 8-6 2-2 6z",
 "square":"M5.5 5h13a.5.5 0 0 1 .5.5v13a.5.5 0 0 1-.5.5h-13a.5.5 0 0 1-.5-.5v-13a.5.5 0 0 1 .5-.5z",
}
ICON_INK = {  # default colour per icon
 **{k: "amber"  for k in ["bulb","star","trophy","coffee","sparkle","learn"]},
 **{k: "red"    for k in ["warn","cross","heart","stop","bug","rocket","webhook","flag"]},
 **{k: "green"  for k in ["check","sheet","money","euro","shield","refresh","target"]},
 **{k: "purple" for k in ["robot","ai","brain","swirl"]},
 **{k: "teal"   for k in ["clock","calendar","wait","db","server","cloud","loop","merge","split","branch","filter","box","gear"]},
}
FILLS = {"bulb": "#fef3c7", "star": "#fde68a", "heart": "#ffe4e6", "trophy": "#fef3c7", "ai": "#ede9fe"}

def icon_svg(name, size="1.3em", colour=None, sw=1.7, extra_style=""):
    c = INK[colour or ICON_INK.get(name, "blue")]
    fill = FILLS.get(name, "none")
    return (f'<svg class="ico" style="width:{size};height:{size};{extra_style}" viewBox="0 0 24 24" fill="{fill}" '
            f'stroke="{c}" stroke-width="{sw}" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">'
            f'<path d="{ICONS[name]}"/></svg>')
print(len(ICONS), "icons loaded:", ", ".join(sorted(ICONS)))


71 icons loaded: ai, arrow, bolt, book, box, brain, branch, bug, bulb, calendar, cart, chart, chat, check, clock, cloud, code, coffee, cross, cursor, db, euro, eye, file, filter, flag, form, gear, globe, hash, heart, home, key, keyboard, learn, link, list, lock, loop, mail, merge, money, pencil, phone, pin, play, plug, question, refresh, robot, rocket, search, send, server, sheet, shield, sparkle, split, square, star, stop, swirl, table, target, trophy, user, users, wait, warn, webhook, wrench


In [ ]:
# ── 5. STYLE: the notebook look (spiral, wobbly ink lines, clouds, stickies) ─
from string import Template
from urllib.parse import quote

def _wave(colour):
    svg = (f"<svg xmlns='http://www.w3.org/2000/svg' width='20' height='7' viewBox='0 0 20 7'>"
           f"<path d='M0 4 Q5 0.8 10 4 T20 4' fill='none' stroke='{colour}' stroke-width='1.6' stroke-linecap='round'/></svg>")
    return "url(\"data:image/svg+xml;utf8," + quote(svg) + "\")"

SPIRAL = "url(\"data:image/svg+xml;utf8," + quote(
    "<svg xmlns='http://www.w3.org/2000/svg' width='52' height='34'>"
    "<circle cx='38' cy='17' r='4.2' fill='#fff' stroke='#555' stroke-width='1.4'/>"
    "<path d='M3 25 C 12 4, 30 3, 38 15' fill='none' stroke='#444' stroke-width='2.6' stroke-linecap='round'/>"
    "<path d='M3 25 C 12 4, 30 3, 38 15' fill='none' stroke='#c8c8c8' stroke-width='0.8' stroke-linecap='round' transform='translate(0.6,-0.8)'/>"
    "</svg>") + "\")"

# SVG filters that make straight CSS borders look hand-drawn
FILTERS = """<svg width="0" height="0" style="position:absolute" aria-hidden="true">
<filter id="rough"><feTurbulence type="fractalNoise" baseFrequency="0.035" numOctaves="2" seed="4"/><feDisplacementMap in="SourceGraphic" scale="3.2"/></filter>
<filter id="rough2"><feTurbulence type="fractalNoise" baseFrequency="0.05" numOctaves="2" seed="11"/><feDisplacementMap in="SourceGraphic" scale="2.6"/></filter>
<filter id="tape"><feTurbulence type="fractalNoise" baseFrequency="0.9" numOctaves="1" seed="2"/><feDisplacementMap in="SourceGraphic" scale="3"/></filter>
</svg>"""

CSS = Template("""
@font-face{font-family:'Ink';font-weight:300;src:url(data:font/woff2;base64,$k300)}
@font-face{font-family:'Ink';font-weight:400;src:url(data:font/woff2;base64,$k400)}
@font-face{font-family:'Ink';font-weight:700;src:url(data:font/woff2;base64,$k700)}
@font-face{font-family:'InkI';unicode-range:U+0049;src:url(data:font/woff2;base64,$d400)}
@page{size:794px 1123px;margin:0}
html,body{margin:0;padding:0;background:#fff}
body{font-family:'InkI','Ink',cursive;color:$ink;font-size:14.5px;line-height:1.38;-webkit-print-color-adjust:exact;print-color-adjust:exact}
.page{width:794px;height:1123px;position:relative;overflow:hidden;background:#fff;break-after:page}
$ruled
.spiral{position:absolute;left:0;top:8px;bottom:8px;width:52px;background:$spiral repeat-y}
.pno{position:absolute;right:34px;bottom:9px;font-size:13px;color:#666;font-weight:700;font-style:italic;z-index:2}
.in{position:absolute;left:68px;right:34px;top:22px;bottom:30px;display:flex;flex-direction:column;gap:7px}
/* title */
.thead{display:flex;align-items:center;justify-content:center;gap:14px;min-height:60px;margin-bottom:2px}
.tbox{position:relative;padding:3px 26px 1px;font-size:28px;font-weight:700;font-variant:small-caps;letter-spacing:.5px;text-align:center;line-height:1.22;max-width:560px}
.tbox::before,.tbox::after{content:'';position:absolute;pointer-events:none}
.tbox::before{inset:0;border:2px solid $ink;filter:url(#rough)}
.tbox::after{inset:-5px;border:1.2px solid $ink;filter:url(#rough2)}
.doodle{flex:none}
/* text */
h2{font-size:18px;font-weight:700;margin:5px 0 2px;padding-bottom:6px;width:fit-content;display:flex;align-items:center;gap:6px;background-repeat:repeat-x;background-position:0 100%;background-size:20px 7px}
h2.c0{color:$blue;background-image:$wblue} h2.c1{color:$teal;background-image:$wteal}
h2.c2{color:$red;background-image:$wred} h2.c3{color:$green;background-image:$wgreen}
h3{font-size:15.5px;margin:3px 0 1px;font-weight:700}
p{margin:0 0 3px}
ul,ol{margin:1px 0 4px;padding-left:22px} li{margin:1px 0}
ul>li::marker{color:$teal;content:'\\2756  ';font-size:.8em}
ol>li::marker{color:$blue;font-weight:700}
li.task{list-style:none;margin-left:-20px;display:flex;gap:6px;align-items:flex-start}
li.task .tb{flex:none;margin-top:2px}
strong{font-weight:700}
em{font-weight:300}
a{color:$blue}
mark{color:inherit;background:linear-gradient(100deg,rgba(255,226,80,0) 1%,rgba(255,226,80,.85) 3%,rgba(255,226,80,.55) 50%,rgba(255,226,80,.8) 96%,rgba(255,226,80,0) 99%);padding:0 3px;margin:0 -1px;border-radius:9px 3px 8px 4px;-webkit-box-decoration-break:clone;box-decoration-break:clone}
.ink-k{color:$pink}.ink-r{color:$red}.ink-b{color:$blue}.ink-t{color:$teal}.ink-g{color:$green}.ink-p{color:$purple}.ink-a{color:$amber}
code{font-family:inherit;font-weight:700;color:$blue}
pre{position:relative;margin:3px 0;padding:6px 11px;${codefill}white-space:pre-wrap;font-size:13.5px;line-height:1.34;color:$blue;font-weight:700}
pre::before{content:'';position:absolute;inset:0;border:1.4px dashed $blue;border-radius:7px;filter:url(#rough2);pointer-events:none}
pre code{color:inherit}
.tk-k{color:$blue;font-weight:700}.tk-f{color:$teal;font-weight:700}.tk-s{color:$green;font-weight:700}
.tk-t{color:$red;font-weight:700}.tk-c{color:#7a7f8c;font-weight:400;font-style:italic}.tk-m{color:$purple;font-weight:700}
/* result chips: ((Bananas 1.20)) */
.rchip{--c:$green;display:inline-block;padding:0 9px;margin:1px 2px;font-weight:700;font-size:.92em;line-height:1.35;color:var(--c);
 background:color-mix(in srgb,var(--c) 11%,#fff);border:1.3px solid var(--c);border-radius:14px;white-space:nowrap}
.rc-b{--c:$blue}.rc-t{--c:$teal}.rc-r{--c:$red}.rc-p{--c:$purple}.rc-a{--c:$amber}.rc-k{--c:$pink}.rc-g{--c:$green}
/* > plain: line */
.plain{margin:2px 0 3px;color:$ink}.plain .pl{font-weight:700;color:$purple;margin-right:4px}
.plain p{display:inline;margin:0}
/* ::: parts : labelled pieces of a statement */
.parts{display:flex;flex-wrap:wrap;gap:6px 14px;align-items:flex-start;margin:4px 0}
.part1{--c:$ink;text-align:center;color:var(--c)}
.part1 .pt{font-size:17px;font-weight:700;padding:0 4px 2px;border-bottom:2.6px solid var(--c);border-radius:0 0 8px 3px}
.part1 .pc{font-size:11.5px;font-style:italic;line-height:1.15;margin-top:2px;white-space:pre-line}
/* ::: chain : stage boxes joined by arrows */
.chain{display:flex;align-items:center;gap:3px;margin:5px 0;flex-wrap:nowrap}
.cn{--c:$ink;flex:1;min-width:0;position:relative;text-align:center;padding:5px 5px 4px;font-size:13px;line-height:1.2;background:color-mix(in srgb,var(--c) 8%,#fff)}
.cn::before{content:'';position:absolute;inset:0;border:1.6px solid var(--c);border-radius:10px 4px 11px 5px;filter:url(#rough)}
.cn b{display:block;font-size:15px;color:var(--c);font-weight:700}
.cn small{display:block;font-size:11.5px;color:#444;margin-top:1px}
.chain.v{flex-direction:column;align-items:stretch;gap:5px}
.chain.v .cn{display:flex;align-items:center;gap:8px;text-align:left;padding:4px 8px}
.chain.v .cn b{display:inline;font-size:14px}.chain.v .cn .cb{flex:none;width:22px;height:22px;border-radius:50%;background:var(--c);color:#fff;font-weight:700;display:flex;align-items:center;justify-content:center;font-size:13px}
.chain .ca{flex:none}
/* modern title block */
.thead.m{display:block;position:relative;min-height:62px;padding:0 70px 0 62px;margin-bottom:4px;--c:$blue}
.thead.m .tnum{position:absolute;left:0;top:10px;width:46px;height:46px;border-radius:50%;background:var(--c);color:#fff;font-size:21px;font-weight:700;display:flex;align-items:center;justify-content:center}
.thead.m .part{font-size:13.5px;font-weight:700;font-style:italic;color:var(--c);line-height:1.1}
.thead.m .tt{font-size:33px;font-weight:700;line-height:1.08;letter-spacing:.2px}
.thead.m .tt .tw{color:var(--c)}
.thead.m .tsub{font-size:14.5px;font-weight:300;margin-top:2px;line-height:1.25}
.thead.m .tdood{position:absolute;right:2px;top:12px}
.thead.m mark{padding:0 6px}
kbd{display:inline-block;position:relative;font-family:inherit;font-size:12px;font-weight:700;line-height:1.3;padding:0 5px;margin:0 1px;border:1.3px solid $ink;border-radius:4px;box-shadow:0 1.6px 0 $ink;background:#fff;vertical-align:1px}
.ico{display:inline-block;vertical-align:-0.25em;flex:none}
hr{border:none;height:10px;margin:3px 0;background:$wink repeat-x 0 50%;background-size:20px 7px;opacity:.55}
img{max-width:100%;border-radius:4px}
/* tables */
table{border-collapse:separate;border-spacing:0;width:100%;font-size:13.5px;line-height:1.26;margin:2px 0}
th,td{position:relative;padding:2px 7px;text-align:left;vertical-align:top}
th{color:$blue;font-weight:700;background:color-mix(in srgb,$blue 11%,#fff)}
td{background:#fff}tbody tr:nth-child(even) td{background:#f5f8ff}
th::after,td::after{content:'';position:absolute;left:0;top:0;right:-1px;bottom:-1px;border:1.15px solid #3a3a3a;filter:url(#rough);pointer-events:none}
td[style*="right"],th[style*="right"]{text-align:right}
/* boxes */
.box{position:relative;padding:7px 12px;margin:2px 0;--c:$ink;border-radius:12px 5px 14px 6px/6px 14px 5px 12px;${boxfill}}
.box.nofill{background:none}
.box::before{content:'';position:absolute;inset:0;border:1.6px solid var(--c);border-radius:12px 5px 14px 6px/6px 14px 5px 12px;filter:url(#rough);pointer-events:none}
.box.dash::before{border-style:dashed}
.box .bt{font-weight:700;color:var(--c);display:flex;align-items:center;gap:6px;margin-bottom:2px;font-size:15px}
.box>p:last-child,.box>ul:last-child{margin-bottom:0}
.k-ink{--c:$ink}.k-blue{--c:$blue}.k-teal{--c:$teal}.k-red{--c:$red}.k-green{--c:$green}.k-purple{--c:$purple}.k-amber{--c:$amber}.k-pink{--c:$pink}
/* sticky note */
.sticky{position:relative;padding:14px 16px 11px;margin:10px 8px 6px;box-shadow:2px 3px 0 rgba(0,0,0,.07);line-height:1.35}
.sticky::before{content:'';position:absolute;top:-10px;left:50%;width:92px;height:22px;margin-left:-46px;background:rgba(250,160,190,.55);transform:rotate(-3deg);filter:url(#tape)}
.sticky .bt{font-weight:700;margin-bottom:2px}
/* quote */
.quote{position:relative;padding:4px 18px 4px 40px;font-size:16px;font-weight:300;color:$teal}
.quote::before{content:'\\201C';position:absolute;left:4px;top:-12px;font-size:52px;font-weight:700;color:$amber}
/* cloud */
.cloud{position:relative;padding:22px 56px 20px;margin:4px 0 0}
.cloud svg.cl{position:absolute;inset:0;width:100%;height:100%}
.cloud .ch .ico{margin:0 4px}
.cloud .ch{position:relative;text-align:center;font-weight:700;color:$red;font-size:16px;text-decoration:underline;text-underline-offset:3px}
.cloud .ct{position:relative;text-align:center;font-size:14.5px}
/* footer */
.foot{margin-top:auto;display:flex;align-items:center;gap:10px;padding:2px 2px 0}
.foot .band{flex:1;position:relative;padding:5px 18px;text-align:center;font-size:14px;font-weight:700;color:$blue}
.foot .band::before{content:'';position:absolute;inset:0;border:1.5px solid $ink;border-radius:24px;filter:url(#rough);pointer-events:none}
/* layout */
.cols{display:flex;gap:14px;align-items:flex-start}.cols>.col{flex:1;min-width:0}
.grid{display:grid;grid-template-columns:repeat(var(--n),minmax(0,1fr));gap:9px}
.center{text-align:center}
/* steps */
.steps{display:flex;align-items:stretch;gap:2px;margin:4px 0}
.step{flex:1;position:relative;padding:20px 8px 7px;text-align:center;font-size:13.5px;line-height:1.28;margin-top:12px}
.step::before{content:'';position:absolute;inset:0;border:1.4px solid $blue;border-radius:10px 4px 12px 5px;filter:url(#rough);pointer-events:none}
.step .sn{position:absolute;top:-13px;left:50%;margin-left:-14px;width:28px;height:28px;border-radius:50%;background:$blue;color:#fff;font-weight:700;display:flex;align-items:center;justify-content:center;font-size:15px}
.step .sh{font-weight:700;color:$blue;margin-bottom:2px}
.sarrow{align-self:center;flex:none;margin-top:12px}
/* flow diagrams (n8n style nodes) */
.flow{display:flex;flex-direction:column;gap:6px;align-items:center;margin:4px 0}
.frow{display:flex;align-items:flex-start;justify-content:center}
.fgroup{display:flex;flex-direction:column;gap:6px}
.fnode{width:86px;height:80px;display:flex;flex-direction:column;align-items:center}
.fbox{width:52px;height:52px;position:relative;display:flex;align-items:center;justify-content:center;background:#fff}
.fbox::before{content:'';position:absolute;inset:0;border:1.7px solid #3a3a3a;border-radius:10px;filter:url(#rough);pointer-events:none}
.fbox.trig::before{border-radius:26px 10px 10px 26px}
.fbox .bolt{position:absolute;left:-9px;top:-7px}
.flab{font-size:12px;text-align:center;line-height:1.15;margin-top:3px;font-weight:700}
.fcap{font-size:12.5px;color:#666;text-align:center}
/* Q&A */
.qa{column-count:2;column-gap:22px}
.qa h3{color:$blue;font-size:14.5px;break-after:avoid;margin-top:0}
.qa p,.qa ul{break-inside:avoid;margin-bottom:7px}
/* cover + contents */
.cover .in{justify-content:center;align-items:center;gap:18px;text-align:center}
.cover .kick{font-size:20px;color:$purple;transform:rotate(-2deg)}
.cover .tbox{font-size:44px;padding:10px 34px 6px;max-width:600px}
.cover .sub{font-size:19px;max-width:560px}
.cover .ring{display:flex;flex-wrap:wrap;justify-content:center;gap:16px;max-width:520px;margin:8px 0}
.cover .swoosh{margin-top:-14px}
.chips{display:flex;flex-wrap:wrap;gap:8px;justify-content:center;max-width:560px}
.chip{position:relative;padding:2px 12px;font-size:14px;font-weight:700;color:$teal}
.chip::before{content:'';position:absolute;inset:0;border:1.4px solid $teal;border-radius:16px;filter:url(#rough2)}
.mynotes{position:relative;margin-top:6px;background:repeating-linear-gradient(to bottom,transparent 0,transparent 27px,#dbe3ef 27px,#dbe3ef 28px)}
.mynotes span{position:absolute;top:-4px;left:0;font-size:13px;color:#9aa3b5;font-weight:700;background:#fff;padding-right:6px}
.toc{columns:1;font-size:15px}
.toc .row{display:flex;align-items:baseline;gap:6px;margin:2px 0}
.toc .row .dots{flex:1;border-bottom:1.6px dotted #aaa;transform:translateY(-4px)}
.toc .row .n{font-weight:700;color:$blue;min-width:22px;text-align:right}
.toc .sec{font-weight:700;color:$teal;margin-top:8px}
$auto
""")

RULED_CSS = (".page::before{content:'';position:absolute;inset:0;pointer-events:none;"
             "background:repeating-linear-gradient(to bottom,transparent 0,transparent 30px,#e4ebf6 30px,#e4ebf6 31px) 0 14px/100% 31px,"
             "linear-gradient(90deg,transparent 61px,#f2b8b8 61px,#f2b8b8 62.6px,transparent 62.6px)}\n"
             ".page>*{z-index:1}.page .spiral{z-index:2}.cover::before{display:none}")

def _auto_css():
    c = [INK[k] for k in ("blue", "teal", "red", "green", "purple", "pink")]
    v = "".join(f".a{i}{{--tc:{col}}}" for i, col in enumerate(c))
    return (v +
      "table[class^=a] th{color:var(--tc);background:color-mix(in srgb,var(--tc) 13%,#fff)}"
      "table[class^=a] td:first-child{font-weight:700;color:var(--tc)}"
      "ul[class^=a]>li::marker,ol[class^=a]>li::marker{color:var(--tc)}"
      "ul[class^=a]>li>strong:first-child,ol[class^=a]>li>strong:first-child{color:var(--tc)}"
      "h2{padding:2px 12px 7px 9px;border-radius:9px 9px 3px 3px}"
      + "".join(f"h2.c{i}{{background-color:color-mix(in srgb,{col} 10%,#fff)}}" for i, col in enumerate(c[:4])))

def build_css():
    auto = _auto_css() if AUTO_COLOUR else ""
    ruled = RULED_CSS if RULED_PAPER else ""
    boxfill = "background:color-mix(in srgb,var(--c) 8%,#fff)" if FILL_BOXES else "background:#fff"
    codefill = "background:color-mix(in srgb,#1d4ed8 6%,#fff);border-radius:8px;" if FILL_BOXES else "background:#fff;"
    return CSS.substitute(**FONTS_B64, **INK, spiral=SPIRAL, ruled=ruled, auto=auto, boxfill=boxfill, codefill=codefill,
        wblue=_wave(INK["blue"]), wteal=_wave(INK["teal"]), wred=_wave(INK["red"]),
        wgreen=_wave(INK["green"]), wink=_wave(INK["ink"]))

CLOUD_SVG = ('<svg class="cl" viewBox="0 0 400 200" preserveAspectRatio="none" aria-hidden="true">'
  '<path d="M60,172 C18,172 8,132 34,116 C8,96 24,54 66,62 C70,24 122,14 146,40 C166,8 226,8 242,38 '
  'C266,12 322,24 322,60 C366,54 392,96 366,118 C392,140 372,182 330,174 C314,196 270,198 250,180 '
  'C226,200 176,200 156,182 C130,200 86,196 60,172 Z" fill="#fffbe3" stroke="#262626" stroke-width="1.6" '
  'vector-effect="non-scaling-stroke" filter="url(#rough2)"/></svg>')
print("Style ready.")


In [ ]:
# ── 6. MARKDOWN → NOTEBOOK PAGES ─────────────────────────────────────────────
import re, html as H, random, yaml
from markdown_it import MarkdownIt
from mdit_py_plugins.container import container_plugin
from bs4 import BeautifulSoup, NavigableString

CONTAINERS = ["cols", "box", "steps", "qa", "grid", "center", "sticky"]

def _container_render(name):
    def render(self, tokens, idx, options, env):
        t = tokens[idx]
        if t.nesting == 1:
            info = t.info.strip()[len(name):].strip()
            return f'<div class="c-{name}" data-info="{H.escape(info, quote=True)}">\n'
        return "</div>\n"
    return render

MD = MarkdownIt("commonmark", {"html": True, "breaks": False}).enable(["table", "strikethrough"])
for _n in CONTAINERS:
    MD.use(container_plugin, name=_n, render=_container_render(_n))

CALLOUTS = {  # > kind: text
    "tip": ("amber", "bulb", "Tip"), "note": ("blue", "pin", "Note"), "warning": ("red", "warn", "Watch out"),
    "example": ("teal", "pencil", "Example"), "info": ("blue", "book", "Good to know"),
    "remember": ("green", "check", "Remember"), "ai": ("purple", "ai", "AI angle"),
    "cost": ("green", "euro", "Cost check"), "pro": ("purple", "star", "Pro move"),
    "try": ("teal", "play", "Try it"),
}

# ---------- inline shortcuts (applied outside code) ----------
def _inline(text):
    text = re.sub(r"==(.+?)==", r"<mark>\1</mark>", text)
    text = re.sub(r"\[\[(.+?)\]\]", r"<kbd>\1</kbd>", text)
    text = re.sub(r"\{([rbtgpak]):([^{}]+)\}", r'<span class="ink-\1">\2</span>', text)
    text = re.sub(r"\(\((?:([rbtgpak]):)?([^()]+?)\)\)", lambda m: f'<span class="rchip rc-{m.group(1) or "g"}">{m.group(2)}</span>', text)
    text = re.sub(r":([a-z][a-z0-9-]*):", lambda m: icon_svg(m.group(1)) if m.group(1) in ICONS else m.group(0), text)
    return text

def preprocess(md_text):
    out, fence = [], False
    for line in md_text.split("\n"):
        if line.lstrip().startswith("```"):
            fence = not fence; out.append(line); continue
        if fence or line.lstrip().startswith(("<", ":::")):
            out.append(line); continue
        parts = re.split(r"(`+[^`]*?`+)", line)          # keep inline code untouched
        out.append("".join(p if p.startswith("`") else _inline(p) for p in parts))
    return "\n".join(out)

# ---------- flow diagrams ----------
NODE_H, GAP, ARROW_W = 80, 6, 52

def _parse_node(s):
    m = re.match(r"^(?:([\w ]+?):\s+)?(\*)?\s*(?:\[([a-z0-9-]+)\])?\s*(.*)$", s.strip())
    lab, trig, ico, txt = m.groups()
    return {"label": lab, "trig": bool(trig), "icon": ico, "text": txt}

def _node_html(n):
    ico = icon_svg(n["icon"], "28px", sw=1.8) if n["icon"] in ICONS else ""
    bolt = icon_svg("bolt", "18px", "amber", 1.6, "background:#fff;border-radius:50%") if n["trig"] else ""
    bolt = bolt.replace('class="ico"', 'class="ico bolt"')
    cls = "fbox trig" if n["trig"] else "fbox"
    return f'<div class="fnode"><div class="{cls}">{bolt}{ico}</div><div class="flab">{_inline(H.escape(n["text"]))}</div></div>'

def _arrow(src_n, dst_n, labels, seed):
    rnd = random.Random(seed)
    hs, hd = src_n * NODE_H + (src_n - 1) * GAP, dst_n * NODE_H + (dst_n - 1) * GAP
    h = max(hs, hd)
    ys = [26 + i * (NODE_H + GAP) for i in range(src_n)]
    yd = [26 + i * (NODE_H + GAP) for i in range(dst_n)]
    pairs = [(ys[0], y) for y in yd] if src_n == 1 else [(y, yd[0]) for y in ys] if dst_n == 1 else list(zip(ys, yd))
    paths, texts = [], []
    for i, (a, b) in enumerate(pairs):
        j = rnd.uniform(-2, 2)
        paths.append(f'<path d="M2 {a} C {ARROW_W*0.45} {a+j}, {ARROW_W*0.55} {b-j}, {ARROW_W-6} {b}"/>'
                     f'<path d="M{ARROW_W-11} {b-4} L{ARROW_W-5} {b} L{ARROW_W-11} {b+4}"/>')
        lab = labels[i] if i < len(labels) else None
        if lab:
            texts.append(f'<text x="{ARROW_W*0.62}" y="{b-6}" text-anchor="middle" font-size="10.5" '
                         f'font-family="Ink" font-weight="700" stroke="#fff" stroke-width="3" paint-order="stroke" fill="{INK["red"]}">{H.escape(lab)}</text>')
    return (f'<svg class="sarrow2" width="{ARROW_W}" height="{h}" viewBox="0 0 {ARROW_W} {h}" fill="none" '
            f'stroke="{INK["ink"]}" stroke-width="1.6" stroke-linecap="round" aria-hidden="true">{"".join(paths)}{"".join(texts)}</svg>')

def render_flow(body, caption, seed=1):
    rows = []
    for r_i, line in enumerate([l for l in body.splitlines() if l.strip()]):
        groups = [[_parse_node(b) for b in step.split("|")] for step in re.split(r"\s*->\s*", line.strip())]
        parts = []
        for i, g in enumerate(groups):
            if i:
                parts.append(_arrow(len(groups[i-1]), len(g), [n["label"] for n in g], seed + r_i * 10 + i))
            parts.append('<div class="fgroup">' + "".join(_node_html(n) for n in g) + "</div>")
        rows.append('<div class="frow">' + "".join(parts) + "</div>")
    cap = f'<div class="fcap">{_inline(H.escape(caption))}</div>' if caption else ""
    return '<div class="flow">' + "".join(rows) + cap + "</div>"

def expand_flows(md_text, seed):
    def rep(m):
        return "\n" + render_flow(m.group(2), (m.group(1) or "").strip(), seed) + "\n"
    return re.sub(r"^:::\s*flow[ \t]*(.*?)\n(.*?)\n:::[ \t]*$", rep, md_text, flags=re.S | re.M)


# ---------- ::: parts  and  ::: chain  (hand-drawn labels and stage boxes) ----------
COLOUR_NAMES = ["blue", "teal", "red", "green", "purple", "amber", "pink", "ink"]
AUTO_BOX = [0]

def _guess_lang(code):
    t = code.strip()
    if re.match(r"(?is)^(select|with|insert|update|delete|create|alter|drop)", t): return "sql"
    if re.search(r"(?i)(CALCULATE|SUMX|FILTER|DIVIDE|VAR|RETURN|DATESYTD|RELATED)\s*\(|:=|^\s*\w[\w ]*=\s*\w+\(", t): return "dax"
    if re.match(r"^\s*(def |import |from \w+ import|print\()", t): return "python"
    if re.match(r"^\s*(const |let |function |\{\s*\")", t): return "js"
    return ""

def render_parts(body):
    out = []
    for line in [l for l in body.splitlines() if l.strip()]:
        bits = [b.strip() for b in line.split("|")]
        txt, cap, col = (bits + ["", ""])[:3]
        col = col if col in COLOUR_NAMES else "ink"
        cap = cap.replace("\\n", "\n")
        out.append(f'<div class="part1 k-{col}"><div class="pt">{txt}</div><div class="pc">{cap}</div></div>')
    return '<div class="parts">' + "".join(out) + "</div>"

def render_chain(body, info):
    vertical = info.strip().startswith("v")
    items = []
    for raw in re.split(r"\n|\s+->\s+", body.strip()):
        raw = raw.strip().lstrip("-").strip()
        if raw:
            items.append(raw)
    arrow = icon_svg("arrow", "20px", "red", 1.6)
    nodes = []
    for i, raw in enumerate(items):
        title, _, small = raw.partition("::")
        col = COLOUR_NAMES[i % 7] if not vertical else COLOUR_NAMES[i % 7]
        m = re.match(r"^\((\w+)\)\s*", title.strip())          # optional (colour) prefix
        if m and m.group(1) in COLOUR_NAMES:
            col, title = m.group(1), title.strip()[m.end():]
        num = f'<span class="cb">{i+1}</span>' if vertical else ""
        sm = f"<small>{small.strip()}</small>" if small.strip() else ""
        nodes.append(f'<div class="cn k-{col}">{num}<span><b>{title.strip()}</b>{sm}</span></div>')
    joined = ("" if vertical else f'<span class="ca">{arrow}</span>').join(nodes)
    return f'<div class="chain{" v" if vertical else ""}">{joined}</div>'

def expand_blocks(md_text):
    def rp(m): return "\n" + render_parts(m.group(1)) + "\n"
    def rc(m): return "\n" + render_chain(m.group(2), m.group(1)) + "\n"
    md_text = re.sub(r"^:::\s*parts[ \t]*\n(.*?)\n:::[ \t]*$", rp, md_text, flags=re.S | re.M)
    md_text = re.sub(r"^:::\s*chain([ \t]*\w*)\n(.*?)\n:::[ \t]*$", rc, md_text, flags=re.S | re.M)
    return md_text

# ---------- syntax colours for fenced code ----------
_KW = {
 "sql": "select from where group by having order limit offset distinct as and or not in between like ilike is null join inner left right full outer cross on using union all intersect except insert into values update set delete create table alter drop add column primary key foreign references default check unique index view case when then else end exists any some with over partition rows range unbounded preceding following current row asc desc top true false cast interval returning if cascade constraint begin commit rollback truncate rename to natural recursive fetch first next only nulls last filter within lateral",
 "dax": "var return define evaluate measure table column order by start at true false in not and or asc desc",
 "python": "def class return if elif else for while in not and or is None True False import from as with try except finally raise lambda yield pass break continue global",
 "js": "const let var function return if else for while in of new class import from export async await try catch finally throw switch case break continue true false null undefined this typeof",
}
_KW["javascript"] = _KW["js"]; _KW["json"] = "true false null"; _KW["py"] = _KW["python"]
_TABLE_AFTER = {"from", "join", "into", "update", "table", "view", "truncate"}

def highlight_code(code, lang):
    lang = (lang or "").lower()
    if lang not in _KW:
        return H.escape(code)
    kws = set(_KW[lang].split())
    com = r"--[^\n]*|//[^\n]*|/\*.*?\*/" if lang in ("sql", "dax") else (r"\#[^\n]*" if lang in ("python", "py") else r"//[^\n]*|/\*.*?\*/")
    rx = re.compile(rf"(?P<c>{com})|(?P<s>'(?:[^'\\]|\\.|'')*'|\"(?:[^\"\\\n]|\\.|\"\")*\")|(?P<m>\[[^\]\n]+\])|(?P<n>\b\d+(?:\.\d+)?\b)|(?P<w>[A-Za-z_][A-Za-z0-9_]*)|(?P<o>\s+|.)", re.S)
    out, prev_kw = [], None
    pos, toks = 0, list(rx.finditer(code))
    for i, m in enumerate(toks):
        k, t = m.lastgroup, m.group()
        e = H.escape(t)
        if k == "c":   out.append(f'<span class="tk-c">{e}</span>')
        elif k == "s": out.append(f'<span class="tk-s">{e}</span>')
        elif k == "n": out.append(f'<span class="tk-s">{e}</span>')
        elif k == "m": out.append(f'<span class="tk-m">{e}</span>' if lang == "dax" else e)
        elif k == "w":
            nxt = code[m.end():m.end() + 1]
            low = t.lower()
            if low in kws:
                out.append(f'<span class="tk-k">{e}</span>'); prev_kw = low; continue
            if nxt == "(":                      out.append(f'<span class="tk-f">{e}</span>')
            elif lang == "sql" and prev_kw in _TABLE_AFTER: out.append(f'<span class="tk-t">{e}</span>')
            elif lang == "dax" and nxt == "[":  out.append(f'<span class="tk-t">{e}</span>')
            else:                               out.append(e)
            prev_kw = None
            continue
        else: out.append(e)
        if k != "o" or t.strip(): prev_kw = None if k != "o" else prev_kw
    return "".join(out)

# ---------- HTML post-processing ----------
def _frag(html_text):
    return BeautifulSoup(html_text, "html.parser")

def _box(soup, colour, icon, title, inner_nodes, dashed=False):
    box = soup.new_tag("div", attrs={"class": f"box k-{colour}" + (" dash" if dashed else "")})
    if title:
        bt = _frag(f'<div class="bt">{icon_svg(icon, "20px", colour) if icon in ICONS else ""}<span>{title}</span></div>')
        box.append(bt)
    for n in inner_nodes:
        box.append(n)
    return box

def postprocess(body_html, page_no, md_dir, sticky_i):
    soup = _frag(body_html)
    # headings: rotate ink colours
    for i, h2 in enumerate(soup.find_all("h2")):
        h2["class"] = f"c{i % 4}"
    if AUTO_COLOUR:   # rotate colours over tables and bullet lists, page by page
        for ti, t in enumerate(soup.find_all("table")):
            t["class"] = f"a{(ti + page_no) % 6}"
        for li_i, lst in enumerate(soup.find_all(["ul", "ol"])):
            lst["class"] = f"a{(li_i * 2 + page_no + 1) % 6}"
    # task lists
    for li in soup.find_all("li"):
        first = li.contents[0] if li.contents else None
        if isinstance(first, NavigableString) and re.match(r"^\[( |x|X)\]\s", first):
            done = first[1] in "xX"
            first.replace_with(first[4:])
            mark = icon_svg("check" if done else "square", "18px", "green" if done else "ink")
            wrap = soup.new_tag("span", attrs={"class": "tt"})
            for c in list(li.children):
                wrap.append(c.extract())
            li.append(_frag(f'<span class="tb">{mark}</span>'))
            li.append(wrap)
            li["class"] = "task"
    # syntax colours in fenced code
    if SYNTAX_COLOURS:
        for code in soup.select("pre > code"):
            lang = next((c[9:] for c in (code.get("class") or []) if c.startswith("language-")), "")
            if not lang and AUTO_COLOUR:
                lang = _guess_lang(code.get_text())
            if lang.lower() in _KW:
                src = code.get_text()
                code.clear()
                holder = _frag("<pre>" + highlight_code(src, lang) + "</pre>").pre   # <pre> keeps the spaces
                for node in list(holder.contents):
                    code.append(node.extract())
    # blockquote callouts
    footer = None
    for bq in soup.find_all("blockquote"):
        p = bq.find("p")
        txt = p.contents[0] if p and p.contents else None
        m = re.match(r'^\s*(\w+)(?:\s*"([^"]+)")?:\s*', str(txt)) if isinstance(txt, NavigableString) else None
        kind = m.group(1).lower() if m else None
        if m:
            txt.replace_with(str(txt)[m.end():])
        inner = [c for c in list(bq.children)]
        if kind == "takeaway":
            text = "".join(str(c) for c in inner)
            bq.replace_with(_frag(f'<div class="cloud">{CLOUD_SVG}<div class="ch">{icon_svg("star","17px","amber")}{m.group(2) or "Key Takeaway"}{icon_svg("star","17px","amber")}</div><div class="ct">{text}</div></div>'))
        elif kind == "footer":
            text = "".join(str(c) for c in inner).replace("<p>", "").replace("</p>", "")
            footer = text
            bq.decompose()
        elif kind == "sticky":
            col = STICKY_COLOURS[sticky_i[0] % len(STICKY_COLOURS)]; sticky_i[0] += 1
            rot = random.Random(page_no * 7 + sticky_i[0]).uniform(-1.8, 1.8)
            title = f'<div class="bt">{m.group(2)}</div>' if m.group(2) else ""
            text = "".join(str(c) for c in inner)
            bq.replace_with(_frag(f'<div class="sticky" style="background:{col};transform:rotate({rot:.2f}deg)">{title}{text}</div>'))
        elif kind == "plain":
            text = "".join(str(c) for c in inner).replace("<p>", "").replace("</p>", "")
            bq.replace_with(_frag(f'<div class="plain"><span class="pl">In plain English:</span>{text}</div>'))
        elif kind == "quote":
            text = "".join(str(c) for c in inner)
            bq.replace_with(_frag(f'<div class="quote">{text}</div>'))
        elif kind in CALLOUTS:
            colour, icon, title = CALLOUTS[kind]
            bq.replace_with(_box(soup, colour, icon, m.group(2) or title, [c.extract() for c in inner]))
        else:
            if m:  # unknown kind: put the word back
                p.insert(0, NavigableString(str(txt)))
            bq.replace_with(_box(soup, "teal", None, None, [c.extract() for c in inner], dashed=True))
    # containers
    for div in soup.find_all("div", class_=re.compile(r"^c-")):
        name = div["class"][0][2:] if isinstance(div["class"], list) else div["class"][2:]
        info = div.get("data-info", "")
        if name == "cols":
            cols, cur = [], []
            for ch in list(div.children):
                if getattr(ch, "name", None) == "hr":
                    cols.append(cur); cur = []
                else:
                    cur.append(ch.extract() if hasattr(ch, "extract") else ch)
            cols.append(cur)
            new = soup.new_tag("div", attrs={"class": "cols"})
            for c in cols:
                col = soup.new_tag("div", attrs={"class": "col"})
                for n in c:
                    col.append(n)
                new.append(col)
            div.replace_with(new)
        elif name == "box":
            mm = re.match(r'^(\w+)?\s*(?:"([^"]*)")?\s*(?::([a-z0-9-]+):)?', info)
            colour = (mm.group(1) or "ink") if mm else "ink"
            if AUTO_COLOUR and colour not in COLOUR_NAMES:      # no colour given: rotate through the inks
                colour = COLOUR_NAMES[AUTO_BOX[0] % 7]; AUTO_BOX[0] += 1
            dashed = "dash" in info
            nofill = "nofill" in info
            colour = colour if f"k-{colour}" in ".k-ink.k-blue.k-teal.k-red.k-green.k-purple.k-amber.k-pink" else "ink"
            title = mm.group(2) if mm else None
            icon = mm.group(3) if mm else None
            newbox = _box(soup, colour, icon, title, [c.extract() for c in list(div.children)], dashed)
            if nofill: newbox["class"] = newbox["class"] + ["nofill"] if isinstance(newbox["class"], list) else newbox["class"] + " nofill"
            div.replace_with(newbox)
        elif name == "grid":
            n = re.match(r"\d+", info); n = n.group(0) if n else "3"
            div["class"] = "grid"; div["style"] = f"--n:{n}"
        elif name == "center":
            div["class"] = "center"
        elif name == "qa":
            div["class"] = "qa"
        elif name == "sticky":
            col = STICKY_COLOURS[sticky_i[0] % len(STICKY_COLOURS)]; sticky_i[0] += 1
            rot = random.Random(page_no * 5 + sticky_i[0]).uniform(-1.8, 1.8)
            div["class"] = "sticky"; div["style"] = f"background:{col};transform:rotate({rot:.2f}deg)"
            if info.strip('" '):
                div.insert(0, _frag(f'<div class="bt">{info.strip(chr(34)+" ")}</div>'))
        elif name == "steps":
            ol = div.find(["ol", "ul"])
            items = ol.find_all("li", recursive=False) if ol else []
            new = soup.new_tag("div", attrs={"class": "steps"})
            arrow = icon_svg("arrow", "22px", "red", 1.6, "transform:rotate(18deg)")
            for i, li in enumerate(items, 1):
                if i > 1:
                    new.append(_frag(f'<span class="sarrow">{arrow}</span>'))
                st = soup.new_tag("div", attrs={"class": "step"})
                st.append(_frag(f'<span class="sn">{i}</span>'))
                strong = li.find("strong")
                if strong and li.contents and li.contents[0] == strong:
                    sh = soup.new_tag("div", attrs={"class": "sh"}); sh.append(strong.extract()); st.append(sh)
                for c in list(li.children):
                    st.append(c.extract())
                new.append(st)
            div.replace_with(new)
        if div.has_attr("data-info") and div.parent is not None:
            del div["data-info"]
    # images relative to the Markdown file
    for img in soup.find_all("img"):
        src = img.get("src", "")
        if src and not re.match(r"^(https?:|data:|file:)", src):
            img["src"] = (md_dir / src).resolve().as_uri()
    return str(soup), footer

# ---------- forgiving nesting: fixes the colon counts of ::: blocks automatically ----------
def normalise_colons(md_text):
    """An outer block needs more colons than the blocks inside it. Claude often gets that wrong, so we recompute it."""
    lines = md_text.replace("\r\n", "\n").split("\n")
    open_re, close_re = re.compile(r"^(:{3,})\s*(\w.*)$"), re.compile(r"^:{3,}\s*$")
    root = {"children": []}; stack, nodes, fence = [root], [], False
    for i, l in enumerate(lines):
        if l.lstrip().startswith("```"): fence = not fence; continue
        if fence: continue
        if close_re.match(l):
            if len(stack) == 1: return md_text            # stray closer: leave the file alone
            stack.pop()["close"] = i
        elif open_re.match(l):
            n = {"open": i, "children": []}; stack[-1]["children"].append(n); stack.append(n); nodes.append(n)
    if len(stack) != 1:
        print("  warning: a ::: block is never closed (check your Markdown); nesting left as is"); return md_text
    def height(n): return 3 if not n["children"] else 1 + max(height(c) for c in n["children"])
    for n in nodes:
        h = height(n); info = re.sub(r"\s+:$", "", open_re.match(lines[n["open"]]).group(2))
        lines[n["open"]] = ":" * h + " " + info; lines[n["close"]] = ":" * h
    return "\n".join(lines)

def split_pages(md_text):
    front = {}
    m = re.match(r"^---\s*\n(.*?)\n---\s*\n", md_text, re.S)
    if m:
        front = yaml.safe_load(m.group(1)) or {}
        md_text = md_text[m.end():]
    pages, cur, fence, section = [], None, False, None
    for line in md_text.split("\n"):
        if line.lstrip().startswith("```"):
            fence = not fence
        if not fence and re.match(r"^#\s", line):
            if cur: pages.append(cur)
            cur = {"title": line[2:].strip(), "body": [], "section": section}
            continue
        if not fence and re.match(r"^<!--\s*section:(.*?)-->", line.strip()):
            section = re.match(r"^<!--\s*section:(.*?)-->", line.strip()).group(1).strip()
            continue
        ms = re.match(r"^<!--\s*sub:(.*?)-->", line.strip())
        if not fence and ms and cur is not None:
            cur["sub"] = ms.group(1).strip(); continue
        if cur is None:
            cur = {"title": "", "body": [], "section": section}
        cur["body"].append(line)
    if cur: pages.append(cur)
    return front, pages
print("Parser ready.")


In [ ]:
# ── 7. BUILD THE PDF ─────────────────────────────────────────────────────────
import threading, asyncio, sys

DOODLES = ["star", "sparkle", "swirl", "heart", "bolt", "bulb", "arrow", "rocket", "coffee", "target"]
DOODLE_INK = ["blue", "teal", "red", "green", "purple", "amber"]
TROPHY = icon_svg("trophy", "34px", "amber", 1.6)
ROCKET = icon_svg("rocket", "34px", "red", 1.6)
SWOOSH = (f'<svg width="380" height="22" viewBox="0 0 380 22" fill="none" stroke="{INK["red"]}" stroke-width="3" stroke-linecap="round" aria-hidden="true">'
          '<path d="M6 14 C 90 4, 200 4, 374 12" filter="url(#rough2)"/><path d="M40 19 C 140 11, 240 12, 330 17" stroke-width="2"/></svg>')

def _doodle(rnd):
    d = rnd.choice(DOODLES)
    return icon_svg(d, f"{rnd.randint(24, 32)}px", rnd.choice(DOODLE_INK), 1.6,
                    f"transform:rotate({rnd.randint(-18, 18)}deg)").replace('class="ico"', 'class="ico doodle"')

def _auto_title(title, col):
    """Colourful titles for free: first word in the part colour, last word highlighted (unless you set your own markers)."""
    if re.search(r"\{[rbtgpak]:|==", title):
        return _inline(H.escape(title))
    words = H.escape(title).split(" ")
    if len(words) >= 3:
        words[-1] = f"<mark>{words[-1]}</mark>"
    if words and len(words[0]) > 1:
        words[0] = f'<span class="tw">{words[0]}</span>'
    return _inline(" ".join(words))

def page_html(title, inner, footer, page_no, default_footer=None, sub=None, section=None, sec_i=0, total=None):
    rnd = random.Random(page_no * 31 + 7)
    col = DOODLE_INK[sec_i % len(DOODLE_INK)]
    if title and TITLE_STYLE == "modern" and page_no:
        part = f'<div class="part">{_inline(H.escape(section))}</div>' if section else ""
        sb = f'<div class="tsub">{_inline(sub)}</div>' if sub else ""
        head = (f'<div class="thead m k-{col}"><div class="tnum">{page_no:02d}</div>{part}'
                f'<div class="tt">{_auto_title(title, col)}</div>{sb}<div class="tdood">{_doodle(rnd)}</div></div>')
    else:
        head = (f'<div class="thead">{_doodle(rnd)}<div class="tbox">{_inline(H.escape(title))}</div>{_doodle(rnd)}</div>'
                if title else "")
    label = (PAGE_LABEL.format(n=page_no, total=total or "") if PAGE_LABEL else str(page_no)) if page_no else ""
    foot_text = footer or default_footer
    foot = f'<div class="foot">{TROPHY}<div class="band">{_inline(foot_text)}</div>{ROCKET}</div>' if foot_text else ""
    return (f'<section class="page"><div class="spiral"></div><div class="in">{head}{inner}{foot}</div>'
            f'<div class="pno">{label}</div></section>')

def cover_html(front):
    icons = front.get("cover_icons", ["bolt", "robot", "mail", "sheet", "webhook", "clock", "ai", "rocket"])
    ring = "".join(icon_svg(i, "44px", sw=1.5) for i in icons if i in ICONS)
    kick = front.get("kicker", "my handwritten notes")
    sub = _inline(front.get("subtitle", ""))
    tag = front.get("tagline")
    sticky = (f'<div class="sticky" style="background:{STICKY_COLOURS[0]};transform:rotate(-1.6deg);max-width:420px">'
              f'{_inline(tag)}</div>') if tag else ""
    rnd = random.Random(99)
    spots = [(x, y) for x in (90, 200, 560, 690) for y in (70, 210, 860, 1000)] + [(110, 520), (700, 520), (400, 90), (400, 1040)]
    scatter = "".join(f'<div style="position:absolute;left:{x+rnd.randint(-25,25)}px;top:{y+rnd.randint(-25,25)}px">{_doodle(rnd)}</div>' for x, y in spots)
    chips = "".join(f'<span class="chip">{_inline(H.escape(c))}</span>' for c in front.get("chips", []))
    chips = f'<div class="chips">{chips}</div>' if chips else ""
    return (f'<section class="page cover"><div class="spiral"></div>{scatter}<div class="in">'
            f'<div class="kick">{H.escape(kick)}</div><div class="tbox">{_inline(H.escape(front.get("title", "Notes")))}</div>'
            f'<div class="swoosh">{SWOOSH}</div><div class="sub">{sub}</div><div class="ring">{ring}</div>{chips}{sticky}</div></section>')

def contents_html(entries, start_no, total=None):
    rows, last = [], None
    for title, sec, n in entries:
        if sec and sec != last:
            rows.append(f'<div class="sec">{_inline(H.escape(sec))}</div>'); last = sec
        rows.append(f'<div class="row"><span>{_inline(H.escape(title))}</span><span class="dots"></span><span class="n">{n}</span></div>')
    cols = (len(rows) > 30)
    body = f'<div class="toc" style="{"columns:2;column-gap:28px" if cols else ""}">{"".join(rows)}</div>'
    return page_html("Contents", body, None, start_no, total=total)

FIT_JS = """() => { const res=[]; document.querySelectorAll('.page').forEach((pg,i)=>{ const el=pg.querySelector('.in');
  const ch=el.clientHeight, cw=el.clientWidth;
  if(el.scrollHeight<=ch+1){ if(!ADD_NOTES || pg.classList.contains('cover')) return;
    const foot=el.querySelector('.foot'); const used=[...el.children].filter(c=>c!==foot).reduce((a,c)=>Math.max(a,c.offsetTop+c.offsetHeight),0);
    const free=(foot?foot.offsetTop:ch)-used-14;
    if(free>130){ const d=document.createElement('div'); d.className='mynotes'; d.style.height=(free-10)+'px';
      d.innerHTML='<span>my notes</span>'; if(foot) el.insertBefore(d,foot); else el.appendChild(d); }
    return; }
  let z=ch/el.scrollHeight;
  for(let k=0;k<14;k++){ el.style.right='auto'; el.style.bottom='auto'; el.style.width=(cw/z)+'px'; el.style.height=(ch/z)+'px'; el.style.left=(68/z)+'px'; el.style.top=(22/z)+'px'; el.style.zoom=z;
    if(el.scrollHeight<=el.clientHeight+1) break; z=z*0.98; }
  res.push([i+1, Math.round(z*100)]); }); return res; }"""

def build(md_path=INPUT_MD, pdf_path=OUTPUT_PDF, preview=PREVIEW_PAGES):
    md_path, pdf_path = Path(md_path), Path(pdf_path)
    front, pages = split_pages(normalise_colons(md_path.read_text(encoding="utf-8")))
    has_cover = bool(front.get("title"))
    offset = 0                               # cover + contents are unnumbered: page 1 = first notes page
    total = len(pages)
    sec_order = []
    for pg in pages:
        if pg["section"] not in sec_order: sec_order.append(pg["section"])
    sticky_i = [0]
    built, entries = [], []
    for i, pg in enumerate(pages, 1):
        no = i + offset
        body = expand_blocks(expand_flows(preprocess("\n".join(pg["body"])), seed=no))
        inner, footer = postprocess(MD.render(body), no, md_path.parent, sticky_i)
        built.append(page_html(pg["title"], inner, footer, no, front.get("footer"),
                               pg.get("sub"), pg["section"], sec_order.index(pg["section"]), total))
        if pg["title"]:
            entries.append((pg["title"], pg["section"], no))
    html_pages = ([cover_html(front), contents_html(entries, 0, total)] if has_cover else []) + built
    doc = (f'<!doctype html><html><head><meta charset="utf-8"><title>{H.escape(front.get("title", "Notes"))}</title>'
           f'<style>{build_css()}</style></head><body>{FILTERS}{"".join(html_pages)}</body></html>')
    html_file = pdf_path.with_suffix(".html")
    html_file.write_text(doc, encoding="utf-8")

    result = {}
    def work():   # runs in its own thread so it also works inside Jupyter / VS Code
        old_policy = asyncio.get_event_loop_policy()
        try:
            if sys.platform == "win32":   # Jupyter on Windows uses a loop type that can't start Chromium
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            from playwright.sync_api import sync_playwright
            with sync_playwright() as p:
                kw = {"executable_path": CHROMIUM_PATH} if CHROMIUM_PATH else {}
                browser = p.chromium.launch(**kw)
                page = browser.new_page(viewport={"width": 794, "height": 1123})
                page.goto(html_file.resolve().as_uri())
                page.evaluate("document.fonts.ready"); page.wait_for_timeout(400)
                result["fitted"] = page.evaluate(FIT_JS.replace("ADD_NOTES", "true" if MY_NOTES_AREA else "false"))
                for n in preview:
                    els = page.query_selector_all(".page")
                    if 0 < n <= len(els):
                        els[n - 1].screenshot(path=str(pdf_path.with_name(f"preview_p{n:02d}.png")))
                page.pdf(path=str(pdf_path), width="794px", height="1123px", print_background=True)
                browser.close()
        except Exception as e:
            result["error"] = e
        finally:
            asyncio.set_event_loop_policy(old_policy)
    t = threading.Thread(target=work); t.start(); t.join()
    if "error" in result:
        raise result["error"]
    print(f"Done: {pdf_path.name} with {len(html_pages)} pages")
    for n, z in result.get("fitted", []):
        print(f"  page {n} was {z}% scaled to fit (split it with a new # title if it looks small)")
    return pdf_path

In [11]:
# ── 8. RUN ───────────────────────────────────────────────────────────────────
build()


KeyboardInterrupt: 